# Embedding & Similarity Demo
**Model:** `all-MiniLM-L6-v2` via `sentence-transformers`  
**Topics:** Cricket · Cooking · Cybersecurity  
**Tasks:** Generate embeddings → 10×10 cosine similarity heatmap → top-2 query results

## 1. Install dependencies

In [ ]:
# Install required packages (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "sentence-transformers", "matplotlib", "seaborn", "numpy"],
                      stdout=subprocess.DEVNULL)

## 2. Define the 10 sentences (3 topics)

In [ ]:
# 10 sentences across exactly 3 topics (≥3 per topic)
sentences = [
    # --- Cricket (4 sentences, indices 0-3) ---
    "The batsman hit a six over the boundary to win the match.",
    "The spinner delivered a googly that completely deceived the batsman.",
    "India chased down a target of 320 runs with two overs to spare.",
    "The fielder took a stunning catch at the boundary to dismiss the opener.",

    # --- Cooking (3 sentences, indices 4-6) ---
    "Sauté the onions over medium heat until they turn golden and translucent.",
    "Marinating the chicken overnight in yoghurt and spices keeps it tender.",
    "A well-seasoned cast iron pan distributes heat evenly for the perfect sear.",

    # --- Cybersecurity (3 sentences, indices 7-9) ---
    "The attacker used a phishing email to steal the employee's login credentials.",
    "Encrypting data at rest ensures sensitive information stays protected if a drive is stolen.",
    "Multi-factor authentication significantly reduces the risk of unauthorised account access.",
]

# Labels for the heatmap axes
labels = [
    "[Cricket] Six to win",
    "[Cricket] Googly delivery",
    "[Cricket] Chase 320",
    "[Cricket] Boundary catch",
    "[Cooking] Sauté onions",
    "[Cooking] Marinate chicken",
    "[Cooking] Cast iron pan",
    "[CyberSec] Phishing email",
    "[CyberSec] Encrypt at rest",
    "[CyberSec] Multi-factor auth",
]

print(f"Loaded {len(sentences)} sentences across 3 topics.")
for i, (lbl, sent) in enumerate(zip(labels, sentences)):
    print(f"  {i:2d}. {lbl}\n      → {sent}")

## 3. Load model and generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)

# Encode all 10 sentences → shape (10, 384)
embeddings = model.encode(sentences, normalize_embeddings=True)

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"Each vector dimension   : {embeddings.shape[1]}")
print(f"Vectors are L2-normalised: {np.allclose(np.linalg.norm(embeddings, axis=1), 1.0)}")

## 4. Compute 10×10 cosine similarity matrix

In [ ]:
# Because vectors are L2-normalised, cosine similarity = dot product
similarity_matrix = np.dot(embeddings, embeddings.T)

print("10×10 Cosine Similarity Matrix")
print("=" * 60)
header = f"{'':25s}" + "".join(f"{i:6d}" for i in range(10))
print(header)
for i, row in enumerate(similarity_matrix):
    row_str = f"{labels[i][:24]:25s}" + "".join(f"{v:6.2f}" for v in row)
    print(row_str)

## 5. Visualise as a heatmap

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Colour scheme ──────────────────────────────────────────────────────────────
TOPIC_COLOURS = {
    "Cricket"      : "#1a6b3a",   # deep green
    "Cooking"      : "#b85c00",   # burnt orange
    "CyberSec"     : "#1a3a6b",   # navy
}
tick_colours = (
    [TOPIC_COLOURS["Cricket"]]  * 4 +
    [TOPIC_COLOURS["Cooking"]]  * 3 +
    [TOPIC_COLOURS["CyberSec"]] * 3
)

# Short axis tick labels (strip topic tag)
short_labels = [
    "Six to win", "Googly delivery", "Chase 320", "Boundary catch",
    "Sauté onions", "Marinate chicken", "Cast iron pan",
    "Phishing email", "Encrypt at rest", "MFA",
]

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor("#f8f6f1")
ax.set_facecolor("#f8f6f1")

sns.heatmap(
    similarity_matrix,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0, vmax=1,
    linewidths=0.5,
    linecolor="#ddd",
    xticklabels=short_labels,
    yticklabels=short_labels,
    cbar_kws={"shrink": 0.75, "label": "Cosine Similarity"},
    annot_kws={"size": 8.5},
)

# Colour tick labels by topic
for tick, colour in zip(ax.get_xticklabels(), tick_colours):
    tick.set_color(colour)
    tick.set_fontweight("bold")
    tick.set_fontsize(8.5)
for tick, colour in zip(ax.get_yticklabels(), tick_colours):
    tick.set_color(colour)
    tick.set_fontweight("bold")
    tick.set_fontsize(8.5)

plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)

# Draw bounding boxes around topic blocks
block_sizes = [4, 3, 3]
block_colours = list(TOPIC_COLOURS.values())
start = 0
for size, colour in zip(block_sizes, block_colours):
    rect = mpatches.FancyBboxPatch(
        (start, start), size, size,
        boxstyle="square,pad=0",
        linewidth=2.5, edgecolor=colour, facecolor="none",
        zorder=5,
    )
    ax.add_patch(rect)
    start += size

# Legend
legend_patches = [
    mpatches.Patch(color=c, label=t)
    for t, c in TOPIC_COLOURS.items()
]
ax.legend(
    handles=legend_patches,
    loc="upper right",
    bbox_to_anchor=(1.28, 1.12),
    framealpha=0.9,
    fontsize=9,
    title="Topic",
    title_fontsize=9,
)

ax.set_title(
    "10 × 10 Cosine Similarity Matrix\nall-MiniLM-L6-v2  |  Cricket · Cooking · Cybersecurity",
    fontsize=13, fontweight="bold", pad=16, color="#1a1a2e",
)

plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Heatmap saved → similarity_heatmap.png")

## 6. Query: find top-2 most similar sentences

In [ ]:
QUERY = "The bowler took three wickets in one over."

# Embed and normalise the query
query_embedding = model.encode([QUERY], normalize_embeddings=True)[0]

# Cosine similarities against all 10 corpus sentences
cos_scores = np.dot(embeddings, query_embedding)   # shape (10,)

# Rank descending; take top-2
top2_idx = np.argsort(cos_scores)[::-1][:2]

print("=" * 62)
print(f'Query : "{QUERY}"')
print("=" * 62)
print(f"{'Rank':<5} {'Score':>6}   Sentence")
print("-" * 62)
for rank, idx in enumerate(top2_idx, start=1):
    print(f"#{rank:<4} {cos_scores[idx]:>6.4f}   {sentences[idx]}")
    print(f"         Topic tag : {labels[idx]}")
    print()
print("=" * 62)
print("\nInterpretation:")
print("  Scores close to 1.0 → semantically near-identical")
print("  Scores close to 0.0 → unrelated topics")
print("  The model correctly surfaces Cricket sentences for a")
print("  cricket-specific query, demonstrating topic clustering.")

## 7. Similarity score summary — all sentences vs query

In [ ]:
# Full ranked list — useful for inspecting inter-topic score gaps
all_ranked = np.argsort(cos_scores)[::-1]

print(f"All 10 sentences ranked by similarity to:\n\"{QUERY}\"\n")
print(f"{'Rank':<5} {'Score':>6}   Label")
print("-" * 55)
for rank, idx in enumerate(all_ranked, start=1):
    print(f"#{rank:<4} {cos_scores[idx]:>6.4f}   {labels[idx]}")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#f8f6f1")
ax.set_facecolor("#f8f6f1")

bar_colours = tick_colours   # already ordered 0-9
bars = ax.barh(
    range(len(sentences)),
    cos_scores,
    color=[bar_colours[i] for i in range(10)],
    edgecolor="white",
    height=0.6,
)
ax.set_yticks(range(len(sentences)))
ax.set_yticklabels(short_labels, fontsize=9)
for tick, colour in zip(ax.get_yticklabels(), tick_colours):
    tick.set_color(colour)
    tick.set_fontweight("bold")

# Annotate scores
for i, (bar, score) in enumerate(zip(bars, cos_scores)):
    ax.text(score + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{score:.4f}", va="center", fontsize=8, color="#333")

ax.set_xlabel("Cosine Similarity", fontsize=10)
ax.set_xlim(0, 1.05)
ax.axvline(0, color="#ccc", lw=0.8)
ax.set_title(
    f'Similarity to query: "{QUERY}"',
    fontsize=11, fontweight="bold", pad=10, color="#1a1a2e",
)
ax.legend(handles=legend_patches, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("query_similarity_bar.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Bar chart saved → query_similarity_bar.png")